# Module 0: How LLMs work

This notebook is the whole exercise. Read each section, then run the code from top to bottom. When you finish, you should be able to explain one request, end to end, and say what each idea is for and where it stops working.

The code cells demonstrate four things:

1. Token counting and a cost estimate.
2. The same prompt at temperature 0 and temperature 1.
3. An unsupported answer, then the same question answered from a trusted note.
4. Semantic ranking of 10 sentences with embeddings.

Stack on this machine: Python, `tiktoken` for token counts, a local MLX model (`Qwen2.5-0.5B-Instruct`, 4-bit) for generation, and `all-MiniLM-L6-v2` for embeddings. That covers the default path (a local model, no hosted Claude account). The same prompts can be sent to any approved hosted chat model if you want a second opinion. Nothing below is written for one provider's SDK.

## The request lifecycle

One chat reply is a loop over tokens. The application builds a single packet of text. The model never sees "a conversation" as a special object. It sees tokens.

1. **Assemble the request.** System instructions, past turns, tool results, retrieved notes, and the new user message are concatenated. This packet has to fit in the context window, with room left for the reply.
2. **Tokenize.** A tokenizer splits the text into token ids. The bill and the window are counted in tokens, so the tokenizer decides the length.
3. **Prefill.** One forward pass reads the whole prompt and builds the model's working state for this request.
4. **Predict the next token.** The model scores every token in its vocabulary. A sampler turns those scores into one choice. Temperature is the knob on that choice.
5. **Append and repeat.** The chosen token is added to the sequence. The model predicts again, until a stop token or the maximum length.
6. **Detokenize.** Token ids become the text you read. You pay for every input token you sent and every output token the model wrote.

Training is not part of this loop. At inference the weights stay frozen. The model is completing the packet, not looking up a database and not remembering earlier chats that you forgot to resend.

## 1. Tokens, length, and cost

A token is a chunk of text the model treats as one unit. It can be a word, part of a word, a space, or a piece of punctuation. `tokenization` is several tokens. A short emoji can be several tokens too.

**Purpose.** Tokens are the unit of the context window, of latency, and of price. A longer prompt costs more because it contains more tokens. A longer answer costs more because the model generates one token at a time.

**Limitation.** Each model family has its own tokenizer. A count from `cl100k_base` (a common OpenAI-family tokenizer, used here because it runs locally with no API key) is a useful estimate. The bill for a real call is the count from that model's tokenizer. Section 3 measures the same sentence with both.

The prices below are round sample rates for the arithmetic, in the neighborhood of a small hosted chat model. Your provider's price list is the one that matters.

In [1]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

# Sample rates so the arithmetic is visible. Swap in the price for the model you actually call.
INPUT_USD_PER_MILLION = 0.15
OUTPUT_USD_PER_MILLION = 0.60


def n_tokens(text: str) -> int:
    return len(enc.encode(text))


def estimate_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        input_tokens / 1_000_000 * INPUT_USD_PER_MILLION
        + output_tokens / 1_000_000 * OUTPUT_USD_PER_MILLION
    )


print("A token is a vocabulary id, not a word.")
samples = [
    "cat",
    "tokenization",
    "I'm happy",
    "🎉",
    "def add(a, b):\n    return a + b\n",
]
for text in samples:
    ids = enc.encode(text)
    print(f"{len(ids):2} tokens  {text!r}")
    print(f"           ids={ids}")

emoji_ids = enc.encode("🎉")
print()
print(f"Decoding those {len(emoji_ids)} ids together returns {enc.decode(emoji_ids)!r}.")
print("Any single id from that group is not the emoji on its own.")
print("The window and the bill follow the ids, so one emoji can cost as much as several words.")

print()
print("Cost grows with copies of the prompt, not with the number of questions you think you asked")
sentence = "Please summarise the refund policy in one sentence."
for copies in (1, 50):
    prompt = " ".join([sentence] * copies)
    prompt_tokens = n_tokens(prompt)
    answer_tokens = 30
    cost = estimate_usd(prompt_tokens, answer_tokens)
    print(
        f"{copies:2} x prompt  input={prompt_tokens:4}  output={answer_tokens:3}  "
        f"estimate=${cost:.6f}"
    )

A token is a vocabulary id, not a word.
 1 tokens  'cat'
           ids=[4719]
 2 tokens  'tokenization'
           ids=[5963, 2065]
 3 tokens  "I'm happy"
           ids=[40, 2846, 6380]
 3 tokens  '🎉'
           ids=[9468, 236, 231]
12 tokens  'def add(a, b):\n    return a + b\n'
           ids=[755, 923, 2948, 11, 293, 997, 262, 471, 264, 489, 293, 198]

Decoding those 3 ids together returns '🎉'.
Any single id from that group is not the emoji on its own.
The window and the bill follow the ids, so one emoji can cost as much as several words.

Cost grows with copies of the prompt, not with the number of questions you think you asked
 1 x prompt  input=  10  output= 30  estimate=$0.000019
50 x prompt  input= 500  output= 30  estimate=$0.000093


## 2. Training lifecycle

Three stages, in order. Each one changes the weights. None of them runs when a user sends a message.

**Pre-training.** The model reads a huge amount of text and practices one objective: predict the next token. This is where grammar, code patterns, and a lot of general knowledge come from. The result is a model that can continue text. It has not been taught to answer you as an assistant.

**Instruction tuning (supervised fine-tuning).** The next dataset is pairs of instructions and good responses. The same next-token objective now teaches the model that a natural continuation of "Answer briefly:" is a brief answer. Format, role, and following directions come from this stage.

**Preference / alignment.** People compare two answers to the same prompt. The model is updated so the preferred answer becomes more likely than the rejected one. The names you will meet are RLHF (a reward model plus reinforcement learning) and DPO (a direct preference loss). This stage shapes helpfulness, refusals, and tone.

**Purpose.** Pre-training builds the raw ability. Instruction tuning makes it follow a request. Preference training picks which kind of answer should win.

**Limitation.** Later stages steer style and behavior. They do not add a fact checker, and they do not turn the model into a database. A well-aligned model can still produce a fluent answer that was never in a trusted source. That is the hallucination section below.

## 3. Inference and next-token prediction

Inference is the lifecycle in section 1, with the weights held fixed.

The model ends each step with a score (a logit) for every token in its vocabulary. Softmax turns scores into probabilities that add up to 1. **Temperature** divides the scores before softmax.

- Temperature approaching 0 puts almost all of the probability on the single highest score. The sampler then takes that token every time. People call this greedy decoding, and it is what temperature 0 means in this notebook.
- Temperature 1 leaves the model's own probabilities alone and draws from them. The same prompt can finish in different ways.
- Temperature above 1 flattens the distribution further, so unlikely tokens show up more often.

**Top-k** keeps only the k highest-scoring tokens. **Top-p** (nucleus sampling) keeps the smallest set of tokens whose probabilities add up to p. Both cut the long tail. They are separate knobs from temperature. This demo changes only temperature, so you can see that knob by itself.

**Purpose.** Temperature and sampling are how you trade repeatability against variety.

**Limitation.** The highest-scoring token is the most likely continuation. It is not a checked fact. Drawing a different token does not look anything up. Low temperature can repeat a wrong answer. High temperature can leave the sensible part of the distribution, especially on a small model. In the city prompt below, temperature 1 can name a large European city that is not a capital.

The local model is `Qwen2.5-0.5B-Instruct` in 4-bit MLX. It is small on purpose: it runs on this Mac, follows short instructions, and still makes the mistakes the later sections depend on. A hosted model will be more fluent. The shape of the lifecycle is the same.

In [2]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler

MODEL_ID = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
model, tokenizer = load(MODEL_ID)
print("loaded", MODEL_ID)

sentence = "The Northwind Arcade staff-room door code is 4419."
print("tiktoken cl100k_base:", n_tokens(sentence))
print("this model's tokenizer:", len(tokenizer.encode(sentence)))
print("Same sentence, two counts. Use the serving model's tokenizer when the count has to match the bill.")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 1830.33it/s]

loaded mlx-community/Qwen2.5-0.5B-Instruct-4bit
tiktoken cl100k_base: 13
this model's tokenizer: 15
Same sentence, two counts. Use the serving model's tokenizer when the count has to match the bill.


In [3]:
import mlx.core as mx

PROMPT = "Name one European capital city. Reply with only the city name."


def complete(user: str, temperature: float, max_tokens: int = 8) -> str:
    messages = [{"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        sampler=make_sampler(temp=temperature),
    ).strip()


print("temperature 0  (greedy, should repeat)")
for run in range(1, 4):
    print(f"  run {run}: {complete(PROMPT, 0.0)}")

print("temperature 1  (draw from the distribution, runs can differ)")
for run in range(1, 6):
    print(f"  run {run}: {complete(PROMPT, 1.0)}")

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": PROMPT}],
    tokenize=False,
    add_generation_prompt=True,
)
logits = model(mx.array(tokenizer.encode(prompt))[None])[0, -1]
greedy_id = int(mx.argmax(logits).item())
print()
print("temperature 0 picks this token every time:", repr(tokenizer.decode([greedy_id])))

scaled = logits - mx.max(logits)
probs = mx.exp(scaled)
probs = probs / mx.sum(probs)
order = mx.argsort(probs)[-6:][::-1]
mx.eval(order, probs)
print("temperature 1 probability of the next token (top 6):")
for index in order.tolist():
    print(f"  {float(probs[index]):6.1%}  {tokenizer.decode([index])!r}")
print("A leading piece such as 'Br' or 'Bar' is one token. Later steps spell the rest of the word.")
print("The top token is only a fraction of the temperature-1 distribution, so repeats can diverge.")

temperature 0  (greedy, should repeat)
  run 1: Paris


  run 2: Paris
  run 3: Paris
temperature 1  (draw from the distribution, runs can differ)


  run 1: Barcelona
  run 2: Brussels


  run 3: Valencia
  run 4: Brussels


  run 5: Madrid

temperature 0 picks this token every time: 'Paris'
temperature 1 probability of the next token (top 6):
   32.3%  'Paris'
   23.3%  'Br'
    9.1%  'Bar'
    8.6%  'Mad'
    7.6%  'London'
    3.6%  'Berlin'
A leading piece such as 'Br' or 'Bar' is one token. Later steps spell the rest of the word.
The top token is only a fraction of the temperature-1 distribution, so repeats can diverge.


## 4. The context window is a budget

The context window is a fixed number of tokens for one request. Everything you want the model to use has to be paid for out of that number:

- system instructions
- conversation history
- retrieved notes and tool results
- the new user message
- tokens you still need for the answer

**Purpose.** The window is the model's whole working memory for this call. If a fact is outside it, this request cannot use that fact.

**Limitation.** A larger window costs more, runs slower, and still has an edge. Past the limit, the call fails or the application drops text. Dropped history is forgotten for this request. Dropped notes cannot ground the answer. Stuffing the window with low-value text also crowds out the note that mattered.

The trim below drops the oldest history first, then notes from the end of the list. That only works if the note you need is at the front. The embedding section is how you choose that order before you pack the window.

The limit below is 256 tokens so the overflow is obvious. The local Qwen model can take far more than that. The arithmetic on a 32,000-token window is the same: add the sections, reserve room for the answer, then drop the lowest-value text until the request fits.

In [4]:
EXAMPLE_LIMIT = 256
RESERVE_FOR_ANSWER = 48

system = "You are a support assistant. Answer from the retrieved notes. Be brief."
history_turns = [
    f"Turn {i}: the user repeated the forgotten laptop password, and the assistant asked a follow-up."
    for i in range(1, 11)
]
retrieved_chunks = [
    "Password reset: on the lock screen, choose Forgot password and follow the email link.",
    "Shipping policy: standard orders leave the warehouse within two business days.",
    "Office hours: the front desk is open from 09:00 to 17:00 on weekdays.",
]
user = "What do I click to reset the laptop password?"


def packet_tokens(history, chunks) -> int:
    parts = [system, *history, *chunks, user]
    return n_tokens("\n".join(parts))


def pack(history, chunks):
    budget = EXAMPLE_LIMIT - RESERVE_FOR_ANSWER
    kept_history = list(history)
    kept_chunks = list(chunks)
    while kept_history and packet_tokens(kept_history, kept_chunks) > budget:
        kept_history.pop(0)
    while kept_chunks and packet_tokens(kept_history, kept_chunks) > budget:
        kept_chunks.pop()
    return kept_history, kept_chunks, budget


full = packet_tokens(history_turns, retrieved_chunks)
print(f"full request: {full} tokens")
print(f"window {EXAMPLE_LIMIT}, reserved for the answer {RESERVE_FOR_ANSWER}, input budget {EXAMPLE_LIMIT - RESERVE_FOR_ANSWER}")
print(f"over budget by {full - (EXAMPLE_LIMIT - RESERVE_FOR_ANSWER)} tokens")

kept_history, kept_chunks, budget = pack(history_turns, retrieved_chunks)
print()
print(f"after trimming: {packet_tokens(kept_history, kept_chunks)} tokens (budget {budget})")
print(f"history turns kept: {len(kept_history)} of {len(history_turns)} (oldest dropped first)")
print(f"notes kept: {len(kept_chunks)} of {len(retrieved_chunks)}")
for chunk in kept_chunks:
    print("  -", chunk)

full request: 276 tokens
window 256, reserved for the answer 48, input budget 208
over budget by 68 tokens

after trimming: 196 tokens (budget 208)
history turns kept: 6 of 10 (oldest dropped first)
notes kept: 3 of 3
  - Password reset: on the lock screen, choose Forgot password and follow the email link.
  - Shipping policy: standard orders leave the warehouse within two business days.
  - Office hours: the front desk is open from 09:00 to 17:00 on weekdays.


## 5. Hallucinations, and grounding

A hallucination here means a fluent answer that is not supported by a source you trust. It happens because the model has to emit a next token anyway. When the prompt does not contain the fact, a plausible invention is still a likely continuation. Pre-training supplies patterns ("door codes look like short numbers"). Alignment can make the wording more careful. Neither stage checks this particular code.

**Grounding** puts a trusted note inside the context window and tells the model to use that note. The next-token distribution is now conditioned on the note, so the code written in the note becomes a likely continuation.

**Purpose.** Grounding is how you turn a completion engine into an answer that is tied to a document you chose.

**Limitation.** The model can ignore the note, misread it, or blend it with other text in the window. If the retriever supplies the wrong note, the model will confidently repeat the wrong note. Grounding reduces unsupported answers. It does not remove the need to check that the note contains the answer and that the reply followed it.

Both calls below use temperature 0. The only change is the trusted note. A deterministic sampler can still be wrong when the fact is missing.

In [5]:
QUESTION = "What is the Northwind Arcade staff-room door code? Reply with only the code."
NOTE = "Trusted note: The Northwind Arcade staff-room door code is 4419."


def answer(system: str, user: str) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return generate(
        model,
        tokenizer,
        prompt=prompt,
        max_tokens=16,
        sampler=make_sampler(temp=0.0),
    ).strip()


unsupported = answer("Answer the question directly.", QUESTION)
grounded_system = (
    "Answer using only the trusted note. "
    "If the note does not contain the answer, say you do not know.\n\n"
    + NOTE
)
grounded = answer(grounded_system, QUESTION)

print("question:    ", QUESTION)
print("unsupported: ", unsupported)
print("grounded:    ", grounded)
print("note says:    4419")

grounded_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": grounded_system},
        {"role": "user", "content": QUESTION},
    ],
    tokenize=False,
    add_generation_prompt=True,
)
print("grounded prompt tokens:", len(tokenizer.encode(grounded_prompt)))
print("Those tokens are the price of putting the trusted fact inside the window.")

question:     What is the Northwind Arcade staff-room door code? Reply with only the code.
unsupported:  1001
grounded:     4419
note says:    4419
grounded prompt tokens: 71
Those tokens are the price of putting the trusted fact inside the window.


## 6. Embeddings

An embedding model maps text to a vector of numbers. Texts with similar meaning land closer together. The usual comparison is cosine similarity: 1 is the same direction, 0 is unrelated.

**Purpose.** You embed a question and a pile of candidate sentences, then sort. The top of that list is what you spend the context budget on. Embeddings are how retrieval chooses the note that the grounding section needs.

**Limitation.** Closeness is similarity, not truth, and not an exact keyword hit. Negation, codes, and dates are easy to mishandle. This MiniLM model also truncates each input at 256 tokens, so a long document has to be split before it is embedded. Vectors from two different embedding models are not comparable.

The query below is about resetting a forgotten laptop password. Ten sentences are ranked against it. The password sentences should rise. The bakery, the train, and the flour should sink.

In [6]:
import numpy as np
from fastembed import TextEmbedding

embedder = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

query = "How do I reset a forgotten laptop password?"
sentences = [
    "Open the sign-in screen and choose the reset-password link for your laptop account.",
    "A forgotten Windows password can be reset from an administrator account.",
    "Use the account recovery page if you cannot log in to the computer.",
    "The bakery on Oak Street opens at 7am on weekdays.",
    "Photosynthesis converts sunlight into chemical energy in plants.",
    "To change your login credentials, verify your identity and set a new passphrase.",
    "The train to Lisbon departs from platform 4.",
    "Enable disk encryption before storing customer data on a notebook computer.",
    "Click Forgot password and follow the emailed reset instructions.",
    "Add two cups of flour and bake for 25 minutes.",
]


def cosine(a, b) -> float:
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


vectors = list(embedder.embed([query, *sentences]))
query_vec, sentence_vecs = vectors[0], vectors[1:]
ranked = sorted(
    ((cosine(query_vec, vec), sentence) for vec, sentence in zip(sentence_vecs, sentences)),
    reverse=True,
)

print(f"vector length: {len(query_vec)}")
print(f"query: {query}")
print()
for rank, (score, sentence) in enumerate(ranked, start=1):
    print(f"{rank:2}. {score:0.3f}  {sentence}")

vector length: 384
query: How do I reset a forgotten laptop password?

 1. 0.776  Open the sign-in screen and choose the reset-password link for your laptop account.
 2. 0.711  A forgotten Windows password can be reset from an administrator account.
 3. 0.671  Click Forgot password and follow the emailed reset instructions.
 4. 0.568  Use the account recovery page if you cannot log in to the computer.
 5. 0.346  Enable disk encryption before storing customer data on a notebook computer.
 6. 0.324  To change your login credentials, verify your identity and set a new passphrase.
 7. 0.093  Add two cups of flour and bake for 25 minutes.
 8. 0.062  Photosynthesis converts sunlight into chemical energy in plants.
 9. 0.013  The bakery on Oak Street opens at 7am on weekdays.
10. -0.104  The train to Lisbon departs from platform 4.


## What you should be able to say

An LLM request is one packet of tokens: system instructions, history, retrieved or tool text, and the new question, all sharing one window. The model scores the next token, a sampler picks one, and that repeats. Temperature 0 always takes the top score. Temperature 1 draws from the model's probabilities. Pre-training taught next-token prediction, instruction tuning taught it to follow directions, and preference training taught it which style of answer should win. None of those stages checks your document at inference time. If the fact is missing from the packet, a fluent guess is still a legal continuation. Putting a trusted note in the window makes the supported answer more likely. Embeddings are how you choose which notes are worth that space, and a high similarity score is not the same thing as a correct answer.